In [1]:
import torch
from torch import nn

In [2]:
net = nn.Sequential(
    nn.Linear(4,8),
    nn.ReLU(),
    nn.Linear(8,1)
) 
X = torch.rand((2,4))
net(X)

tensor([[0.2180],
        [0.0043]], grad_fn=<AddmmBackward0>)

In [3]:
## 访问某一层的参数
print(net[2].state_dict())

OrderedDict([('weight', tensor([[-0.1096,  0.1181,  0.1356, -0.2170, -0.2552, -0.1664,  0.1960, -0.3143]])), ('bias', tensor([0.2584]))])


In [4]:
print(type(net[2].bias))
print(net[2].bias)
print(net[2].bias.data)

<class 'torch.nn.parameter.Parameter'>
Parameter containing:
tensor([0.2584], requires_grad=True)
tensor([0.2584])


In [5]:
## 可以访问梯度
net[2].weight.grad

In [6]:
## 通过named_parameters 遍历网络访问所有参数
print(*[(name, param.shape) for name, param in net[0].named_parameters()])
print(*[(name, param.shape) for name, param in net.named_parameters()])

('weight', torch.Size([8, 4])) ('bias', torch.Size([8]))
('0.weight', torch.Size([8, 4])) ('0.bias', torch.Size([8])) ('2.weight', torch.Size([1, 8])) ('2.bias', torch.Size([1]))


In [16]:
## block的嵌套
def block1():
    return nn.Sequential(
        nn.Linear(4,8),
        nn.ReLU(),
        nn.Linear(8,4),
        nn.ReLU()
    )

def block2():
    net = nn.Sequential()
    for i in range(4):
        net.add_module(f'block {i}',block1())
    return net

rgnet = nn.Sequential(
    block2(),
    nn.Linear(4,1)
)

rgnet[0][1][0].bias.detach()

tensor([ 0.0151, -0.3761, -0.1338, -0.0770,  0.2470,  0.1390,  0.2655, -0.1953])

In [19]:
## 用均值为0，std为0.01的高斯分布初始化weight，bias默认设置为0
def init_normal(m):
    if m == nn.Linear:
        nn.init.normal(m.weight,mean=0,std=0.01)
        nn.init.zeros_(m.bias)
net.apply(init_normal)

net[0].weight.data[0]

tensor([-0.1648, -0.4517,  0.4668, -0.2199])

In [21]:
## 将weight初始化为1
def init_constant(m):
    if m == nn.Linear:
        nn.init.constant_(m.weight,1)
        nn.init.zeros_(m.bias)
net.apply(init_constant)

net[0].weight.data[0]

tensor([-0.1648, -0.4517,  0.4668, -0.2199])

In [22]:
def init_xavier(m):
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)
def init_42(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 42)
net[0].apply(init_xavier)
net[2].apply(init_42)
print(net[0].weight.data[0])
print(net[2].weight.data)

tensor([-0.6971, -0.1372, -0.3979, -0.1384])
tensor([[42., 42., 42., 42., 42., 42., 42., 42.]])


In [25]:
## 自定义初始化
def my_init(m):
    if type(m) == nn.Linear:
        print(
            "init", *[(name,param.shape) for name,param in m.named_parameters()][0]
        )
        nn.init.uniform_(m.weight,-10,10)
        m.weight.data *= m.weight.data.abs() >= 5
net.apply(my_init)
net[0].weight[:2]

init weight torch.Size([8, 4])
init weight torch.Size([1, 8])


tensor([[ 6.0155,  0.0000, -9.3991,  9.5956],
        [ 0.0000,  0.0000,  0.0000, -6.9724]], grad_fn=<SliceBackward0>)

In [27]:
shared = nn.Linear(8, 8)

net = nn.Sequential(
    nn.Linear(4, 8), nn.ReLU(),
    shared, nn.ReLU(),
    shared, nn.ReLU(),
    nn.Linear(8, 1)
)

print(net[2].weight.data[0] == net[4].weight.data[0])

tensor([True, True, True, True, True, True, True, True])


In [ ]:
## lazy initialization

net = nn.Sequential(
    nn.LazyLinear(128),
    nn.ReLU(),
    nn.LazyLinear(64),
    nn.ReLU(),
    nn.LazyLinear(10)
)


print(type(net[0].weight))  

# 第一次前向传播，自动初始化
X = torch.randn(4, 20)
Y = net(X)


print(net[0].weight.shape)  # torch.Size([128, 20])
print(net[2].weight.shape)  # torch.Size([64, 128])
print(net[4].weight.shape)  # torch.Size([10, 64])

<class 'torch.nn.parameter.UninitializedParameter'>
torch.Size([128, 20])
torch.Size([64, 128])
torch.Size([10, 64])
